# Baseline Round 1 On 15 Preprocessed Fields

Train `LinearRegression` and `Ridge(alpha=1.0)` on `processed_train.csv`, then compare holdout metrics on both `SalePriceLog` and original price scale.

Known limitations for this round:
- The processed CSV was scale/encoded before this holdout split, so preprocessing statistics have mild leakage into the holdout set.
- This uses one 20% holdout split on about 1460 rows, so results can be noisy with respect to `random_state`.
- Round 2 should consider fitting preprocessing on split-train only and adding k-fold CV for stronger model comparison.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd()
if (cwd / "my-project").exists():
    project_dir = cwd / "my-project"
elif cwd.name == "my-project":
    project_dir = cwd
else:
    project_dir = next(parent for parent in cwd.parents if parent.name == "my-project")

src_dir = project_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from ml.baseline_round1 import (  # noqa: E402
    DEFAULT_ARTIFACTS_DIR,
    DEFAULT_DATA_PATH,
    evaluate_regression_model,
    load_processed_training_data,
    run_baseline_round1,
    split_processed_data,
    train_baseline_models,
)

pd.set_option("display.max_columns", 30)

## Train And Compare

In [ ]:
metrics = run_baseline_round1(
    data_path=DEFAULT_DATA_PATH,
    artifacts_dir=DEFAULT_ARTIFACTS_DIR,
    test_size=0.2,
    random_state=42,
    ridge_alpha=1.0,
)

metrics.sort_values("rmse_log")

In [ ]:
best_row = metrics.sort_values("rmse_log").iloc[0]
print(f"Best baseline by rmse_log: {best_row['model_name']} ({best_row['rmse_log']:.4f})")

## Residual Checks

In [ ]:
x, y = load_processed_training_data(DEFAULT_DATA_PATH)
x_train, x_test, y_train, y_test = split_processed_data(
    x,
    y,
    test_size=0.2,
    random_state=42,
)
models = train_baseline_models(x_train, y_train, ridge_alpha=1.0)
best_model_name = metrics.sort_values("rmse_log").iloc[0]["model_name"]
best_model = models[best_model_name]
y_pred_log = best_model.predict(x_test)

residuals = y_test.to_numpy() - y_pred_log
residual_frame = x_test.copy()
residual_frame["prediction_log"] = y_pred_log
residual_frame["residual_log"] = residuals
residual_frame["actual_price"] = np.expm1(y_test.to_numpy())
residual_frame["predicted_price"] = np.expm1(y_pred_log)
residual_frame.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

axes[0].scatter(residual_frame["prediction_log"], residual_frame["residual_log"], alpha=0.7)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Residuals vs prediction")
axes[0].set_xlabel("Predicted SalePriceLog")
axes[0].set_ylabel("Residual log")

axes[1].hist(residual_frame["residual_log"], bins=30)
axes[1].set_title("Residual distribution")
axes[1].set_xlabel("Residual log")

for axis, column in zip(axes[2:], ["num__GrLivArea", "num__OverallQual"]):
    if column in residual_frame.columns:
        axis.scatter(residual_frame[column], residual_frame["residual_log"], alpha=0.7)
        axis.axhline(0, color="black", linewidth=1)
        axis.set_title(f"Residuals vs {column}")
        axis.set_xlabel(column)
        axis.set_ylabel("Residual log")
    else:
        axis.axis("off")

fig.suptitle(f"Residual checks for {best_model_name}")
fig.tight_layout()
plt.show()

## Notes For Next EDA/Preprocessing Round

- Re-check residual patterns by raw `GrLivArea`, `OverallQual`, and `Neighborhood`.
- Inspect large-area, low-price outliers and decide whether to cap, remove, or model them separately.
- Consider log-transforming skewed numeric features in a leakage-safe train-only preprocessing flow.
- Consider feature interactions or engineered features such as total square footage and house age in round 2.
- Add k-fold CV when comparing serious model candidates beyond this first baseline.